In [0]:
-- Create a new schema with the name ecom_data in the workspace

CREATE SCHEMA IF NOT EXISTS workspace.ecom_data;

In [0]:
-- Show tables in the new schema

SHOW TABLES IN workspace.ecom_data;

database,tableName,isTemporary
ecom_data,clean_orders,false
ecom_data,cohort_customers,false
ecom_data,raw_orders,false
ecom_data,src_orders_2024,false
ecom_data,src_orders_2025,false


In [0]:
/* Upload the file ecommerce...2024_rebalanced.csv in a created volume cohorts in the created schema ecom_data. Create directly this table inthe volume eco_sales a Delta Table with the name src_orders_2024 in the new schema ecom_data */

CREATE OR REPLACE TABLE workspace.ecom_data.src_orders_2024
USING DELTA
AS SELECT * FROM read_files ('/Volumes/workspace/ecom_data/cohorts/ecom_orders_2024_rebalanced - ecom_orders_2024_rebalanced.csv', format => 'csv', header => true, inferSchema => true);

num_affected_rows,num_inserted_rows


In [0]:
DESCRIBE TABLE EXTENDED workspace.ecom_data.src_orders_2024;

col_name,data_type,comment
row_id,int,null
customer_id,string,null
order_date,date,null
order_id,string,null
sales,double,null
_rescued_data,string,null
,,
# Delta Statistics Columns,,
Column Names,"order_id, customer_id, row_id, sales, order_date, _rescued_data",
Column Selection Method,first-32,


In [0]:
-- Count rows and confirm min/max dates are in 2024

SELECT 
  COUNT (*),
  MIN (order_date) AS min_order_date,
  MAX (order_date) AS max_order_date
FROM workspace.ecom_data.src_orders_2024;


COUNT(*),min_order_date,max_order_date
1000,2024-01-01,2024-12-30


In [0]:
%python

# Create a Spark Dataframe from the raw data table

dfraw = spark.table("workspace.ecom_data.src_orders_2024")
dfraw.display()


row_id,customer_id,order_date,order_id,sales,_rescued_data
241,CUST546,2024-01-01,ORD1240,157.06,null
515,CUST260,2024-01-01,ORD1514,79.44,null
239,CUST303,2024-01-01,ORD1238,261.57,null
214,CUST507,2024-01-01,ORD1213,177.22,null
823,CUST411,2024-01-01,ORD1822,140.94,null
26,CUST266,2024-01-02,ORD1025,34.46,null
566,CUST322,2024-01-02,ORD1565,240.26,null
27,CUST522,2024-01-02,ORD1026,189.66,null
713,CUST467,2024-01-02,ORD1712,179.93,null
382,CUST648,2024-01-02,ORD1381,164.95,null


In [0]:
%python

# Create a Bronze table called raw_orders

bronze_path = '/Volumes/workspace.ecom_data.raw_orders/'

In [0]:
%python
# Create the raw Delta table directly in the schema as Bronze-Layer with the table name raw_orders

dfraw = spark.table("workspace.ecom_data.src_orders_2024")
dfraw.write.format('delta').mode('overwrite').saveAsTable('workspace.ecom_data.raw_orders')
dfraw.display()

row_id,customer_id,order_date,order_id,sales,_rescued_data
241,CUST546,2024-01-01,ORD1240,157.06,null
515,CUST260,2024-01-01,ORD1514,79.44,null
239,CUST303,2024-01-01,ORD1238,261.57,null
214,CUST507,2024-01-01,ORD1213,177.22,null
823,CUST411,2024-01-01,ORD1822,140.94,null
26,CUST266,2024-01-02,ORD1025,34.46,null
566,CUST322,2024-01-02,ORD1565,240.26,null
27,CUST522,2024-01-02,ORD1026,189.66,null
713,CUST467,2024-01-02,ORD1712,179.93,null
382,CUST648,2024-01-02,ORD1381,164.95,null


In [0]:
%python

# Standardize formatting while reading Bronze-Layer from Delta table in workspace.ecom_data schema
from pyspark.sql.functions import trim, col, cast

dfraw = spark.table('workspace.ecom_data.raw_orders')

# remove spaces in the columns customer_id and order_id and ensure numeric/decimal format in the column sales
df_bronze = dfraw.withColumn("customer_id", trim(col("customer_id"))) \
    .withColumn("order_id", trim(col("order_id"))) \
    .withColumn("sales", col("sales").cast("decimal(18,2)"))

df_bronze.display()

row_id,customer_id,order_date,order_id,sales,_rescued_data
241,CUST546,2024-01-01,ORD1240,157.06,null
515,CUST260,2024-01-01,ORD1514,79.44,null
239,CUST303,2024-01-01,ORD1238,261.57,null
214,CUST507,2024-01-01,ORD1213,177.22,null
823,CUST411,2024-01-01,ORD1822,140.94,null
26,CUST266,2024-01-02,ORD1025,34.46,null
566,CUST322,2024-01-02,ORD1565,240.26,null
27,CUST522,2024-01-02,ORD1026,189.66,null
713,CUST467,2024-01-02,ORD1712,179.93,null
382,CUST648,2024-01-02,ORD1381,164.95,null


In [0]:
%python

# Check the Bronze layer:
# total rows
# # unique orders
# # unique customers
# min/max order date
# Hint:
# Use COUNT(*), COUNT(DISTINCT ...), MIN(...), MAX(...). */

from pyspark.sql import functions as F

# Load Bronze-Layer as Spark DataFrame 
df_bronze = spark.table('workspace.ecom_data.raw_orders')

# Check of the Key figures
summary = df_bronze.agg(
    F.count("*").alias("total_rows"),
    F.countDistinct("customer_id").alias("unique_customers"),
    F.countDistinct("order_id").alias("unique_orders"),
    F.min("order_date").alias("min_order_date"),
    F.max("order_date").alias("max_order_date")
)

display(summary)




total_rows,unique_customers,unique_orders,min_order_date,max_order_date
1000,649,1000,2024-01-01,2024-12-30


In [0]:
%python

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Load Bronze table
df_bronze = spark.table('workspace.ecom_data.raw_orders')

# Remove invalid rows
cleaned = df_bronze.filter(
    (F.col('customer_id').isNotNull()) &
    (F.col('order_id').isNotNull()) &
    (F.col('order_date').isNotNull()) &
    (F.col('sales').isNotNull()) &
    (F.col('sales') > 0)
)

# Deduplicate orders using window function
window = Window.partitionBy('order_id').orderBy(F.col('order_date').asc())
cleaned = cleaned.withColumn('row_num', F.row_number().over(window))
deduped = cleaned.filter(F.col('row_num') == 1).drop('row_num')

# Save the depuded data as a new Delta table in the schema as Silver--Layer with the table name clean_orders

deduped.write.format('delta').mode('overwrite').saveAsTable('workspace.ecom_data.clean_orders')
deduped.display()

row_id,customer_id,order_date,order_id,sales,_rescued_data
1,CUST272,2024-05-17,ORD1000,296.26,null
2,CUST151,2024-01-02,ORD1001,286.33,null
3,CUST554,2024-09-18,ORD1002,53.17,null
4,CUST061,2024-04-08,ORD1003,198.38,null
5,CUST578,2024-11-06,ORD1004,54.14,null
6,CUST188,2024-06-23,ORD1005,212.11,null
7,CUST670,2024-08-02,ORD1006,23.44,null
8,CUST353,2024-02-04,ORD1007,251.11,null
9,CUST039,2024-06-14,ORD1008,102.7,null
10,CUST253,2024-12-19,ORD1009,143.85,null


In [0]:
%python
from pyspark.sql import functions as F

# Load Silver table

df_silver = spark.table('workspace.ecom_data.clean_orders')

# Check summary statistics
summary = df_silver.agg(
    F.count("*").alias("rows"),
    F.countDistinct("order_id").alias("unique_orders"),
    F.countDistinct("customer_id").alias("customers"),
    F.min("order_date").alias("min_order_date"),
    F.max("order_date").alias("max_order_date")
)
display(summary)

rows,unique_orders,customers,min_order_date,max_order_date
1000,1000,649,2024-01-01,2024-12-30


In [0]:
-- Final check: Use SHOW TABLES to confirm the tables exist.

SHOW TABLES in workspace.ecom_data;

database,tableName,isTemporary
ecom_data,clean_orders,false
ecom_data,cohort_customers,false
ecom_data,raw_orders,false
ecom_data,src_orders_2024,false
ecom_data,src_orders_2025,false


In [0]:
-- Preview clean_orders with LIMIT 10 to confirm columns look correct

SELECT * FROM workspace.ecom_data.clean_orders LIMIT 10;

row_id,customer_id,order_date,order_id,sales,_rescued_data
1,CUST272,2024-05-17,ORD1000,296.26,null
2,CUST151,2024-01-02,ORD1001,286.33,null
3,CUST554,2024-09-18,ORD1002,53.17,null
4,CUST061,2024-04-08,ORD1003,198.38,null
5,CUST578,2024-11-06,ORD1004,54.14,null
6,CUST188,2024-06-23,ORD1005,212.11,null
7,CUST670,2024-08-02,ORD1006,23.44,null
8,CUST353,2024-02-04,ORD1007,251.11,null
9,CUST039,2024-06-14,ORD1008,102.7,null
10,CUST253,2024-12-19,ORD1009,143.85,null


In [0]:
%python

# Create a Gold-Layer with the Delta Table and the name cohort_customers directly in the schema ecom_data

# We need for this table: Each customer’s first purchase date
# Each customer’s second purchase date (if any): # the earliest order date after the first purchase
# The number of days between the first and second # purchases
# The cohort month (month of first purchase)


from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_silver = spark.table('workspace.ecom_data.clean_orders')

# assign a unique, sequential number to each order belonging to a specific customer, ordered by the date the purchase was made.

w = Window.partitionBy('customer_id').orderBy('order_date')

# The function F.row_number () assigns the number 1 to the earliest order, 2 to the second, and so on.
orders = df_silver.withColumn('row_num', F.row_number().over(w))

# Since I previously ordered the window by order_date, the row with row_num == 1 is mathematically the earliest purchase for that specific customer and keep only the column which is necessary for this analysis, the customer_id as primary key and renamed these first order dates to first_purchase_date

first_orders = orders.filter(F.col('row_num') == 1).select(
    'customer_id',
    F.col('order_date').alias('first_purchase_date')
)
# It does the same with the second_purchase_date, see above and take the row_num == 2

second_orders = orders.filter(F.col('row_num') == 2).select(
    'customer_id',
    F.col('order_date').alias('second_purchase_date')
)
# Use Left Join for the two tables

cohort = first_orders.join(second_orders, 'customer_id', 'left')

# take the first purchase date as new column cohort_month and it should have only the year and month with the date_trunc function

cohort = cohort.withColumn(
    'cohort_month',
    F.date_trunc('month', F.col('first_purchase_date'))
)

# Calculate the difference between the first and second purchase with the datediff function 

cohort = cohort.withColumn(
    'days_to_second_purchase',
    F.datediff('second_purchase_date', 'first_purchase_date')
)

cohort.write.format('delta').mode('overwrite').saveAsTable('workspace.ecom_data.cohort_customers')
cohort.display()

customer_id,first_purchase_date,second_purchase_date,cohort_month,days_to_second_purchase
CUST001,2024-04-27,null,2024-04-01T00:00:00.000Z,null
CUST002,2024-06-05,2024-10-31,2024-06-01T00:00:00.000Z,148
CUST003,2024-07-25,2024-10-13,2024-07-01T00:00:00.000Z,80
CUST004,2024-09-24,null,2024-09-01T00:00:00.000Z,null
CUST007,2024-02-14,null,2024-02-01T00:00:00.000Z,null
CUST008,2024-02-17,2024-11-20,2024-02-01T00:00:00.000Z,277
CUST010,2024-11-05,2024-12-10,2024-11-01T00:00:00.000Z,35
CUST011,2024-10-10,null,2024-10-01T00:00:00.000Z,null
CUST012,2024-03-14,null,2024-03-01T00:00:00.000Z,null
CUST013,2024-02-23,2024-03-19,2024-02-01T00:00:00.000Z,25


In [0]:
-- Sample rows
SELECT * FROM workspace.ecom_data.cohort_customers LIMIT 10;

customer_id,first_purchase_date,second_purchase_date,cohort_month,days_to_second_purchase
CUST001,2024-04-27,null,2024-04-01T00:00:00.000Z,null
CUST002,2024-06-05,2024-10-31,2024-06-01T00:00:00.000Z,148
CUST003,2024-07-25,2024-10-13,2024-07-01T00:00:00.000Z,80
CUST004,2024-09-24,null,2024-09-01T00:00:00.000Z,null
CUST007,2024-02-14,null,2024-02-01T00:00:00.000Z,null
CUST008,2024-02-17,2024-11-20,2024-02-01T00:00:00.000Z,277
CUST010,2024-11-05,2024-12-10,2024-11-01T00:00:00.000Z,35
CUST011,2024-10-10,null,2024-10-01T00:00:00.000Z,null
CUST012,2024-03-14,null,2024-03-01T00:00:00.000Z,null
CUST013,2024-02-23,2024-03-19,2024-02-01T00:00:00.000Z,25


In [0]:
-- Validate: Total customers, repeaters, one-time customers

SELECT
  COUNT(*) AS total_customers,
  COUNT(CASE WHEN second_purchase_date IS NOT NULL THEN 1 END) AS repeaters,
  COUNT(CASE WHEN second_purchase_date IS NULL THEN 1 END) AS one_time_customers
FROM workspace.ecom_data.cohort_customers;


total_customers,repeaters,one_time_customers
649,184,465


In [0]:
/* Create directly a new renamed Delta table src_orders_2025 in the schema ecom_data from the new uploaded file ecommcerc...2025_rebalanced.csv in the volume cohorts */

CREATE OR REPLACE TABLE workspace.ecom_data.src_orders_2025
USING DELTA
AS SELECT * FROM read_files ('/Volumes/workspace/ecom_data/cohorts/ecom_orders_incremental_2025_rebalanced - ecom_orders_incremental_2025_rebalanced.csv', format => 'csv', header => true, inferSchema => true)

num_affected_rows,num_inserted_rows


In [0]:
-- Count rows and confirm min/max dates are in 2025

SELECT 
  COUNT (*),
  MIN (order_date) AS min_order_date,
  MAX (order_date) AS max_order_date
FROM workspace.ecom_data.src_orders_2025;

COUNT(*),min_order_date,max_order_date
1000,2025-01-01,2025-12-31


In [0]:
-- Append rows from src_orders_2025 into your existing Bronze table raw_orders

INSERT INTO workspace.ecom_data.raw_orders (
  row_id,
  customer_id,
  order_date,
  order_id,
  sales

)
 
SELECT 
  row_id,
  customer_id,
  order_date,
  order_id,
  sales
FROM workspace.ecom_data.src_orders_2025;

num_affected_rows,num_inserted_rows
1000,1000


In [0]:
%python
# Standardize formatting with data from 2025 while reading Bronze-Layer from Delta table in workspace.ecom_data schema
from pyspark.sql.functions import trim, col, cast

dfraw = spark.table('workspace.ecom_data.raw_orders')

# remove spaces in the columns customer_id and order_id and ensure numeric/decimal format in the column sales
df_bronze = dfraw.withColumn("customer_id", trim(col("customer_id"))) \
    .withColumn("order_id", trim(col("order_id"))) \
    .withColumn("sales", col("sales").cast("decimal(18,2)"))

df_bronze.display()

row_id,customer_id,order_date,order_id,sales,_rescued_data
241,CUST546,2024-01-01,ORD1240,157.06,null
515,CUST260,2024-01-01,ORD1514,79.44,null
239,CUST303,2024-01-01,ORD1238,261.57,null
214,CUST507,2024-01-01,ORD1213,177.22,null
823,CUST411,2024-01-01,ORD1822,140.94,null
26,CUST266,2024-01-02,ORD1025,34.46,null
566,CUST322,2024-01-02,ORD1565,240.26,null
27,CUST522,2024-01-02,ORD1026,189.66,null
713,CUST467,2024-01-02,ORD1712,179.93,null
382,CUST648,2024-01-02,ORD1381,164.95,null


In [0]:
-- Count rows and confirm min/max dates are in 2024 - 2025

SELECT 
  COUNT(*) AS total_rows,
  MIN(order_date) AS min_order_date,
  MAX(order_date) AS max_order_date
FROM workspace.ecom_data.raw_orders;

total_rows,min_order_date,max_order_date
2000,2024-01-01,2025-12-31


In [0]:
%python

# Check the Bronze layer with the new data from 2025 again:
# total rows
# # unique orders
# # unique customers
# min/max order date
# Hint:
# Use COUNT(*), COUNT(DISTINCT ...), MIN(...), MAX(...). */

from pyspark.sql import functions as F

# Load Bronze-Layer as Spark DataFrame 
df_bronze = spark.table('workspace.ecom_data.raw_orders')

# Check of the Key figures
summary = df_bronze.agg(
    F.count("*").alias("total_rows"),
    F.countDistinct("customer_id").alias("unique_customers"),
    F.countDistinct("order_id").alias("unique_orders"),
    F.min("order_date").alias("min_order_date"),
    F.max("order_date").alias("max_order_date")
)

display(summary)


total_rows,unique_customers,unique_orders,min_order_date,max_order_date
2000,888,2000,2024-01-01,2025-12-31


In [0]:
%python

# Load the raw_orders table with new data from 2025 and verify the same columns if there are NULL-Values and duplicates 

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Load Bronze table
df_bronze = spark.table('workspace.ecom_data.raw_orders')

# Remove invalid rows
cleaned = df_bronze.filter(
    (F.col('customer_id').isNotNull()) &
    (F.col('order_id').isNotNull()) &
    (F.col('order_date').isNotNull()) &
    (F.col('sales').isNotNull()) &
    (F.col('sales') > 0)
)

# Deduplicate orders using window function
window = Window.partitionBy('order_id').orderBy(F.col('order_date').asc())
cleaned = cleaned.withColumn('row_num', F.row_number().over(window))
deduped = cleaned.filter(F.col('row_num') == 1).drop('row_num')

# Save the depuded data as a new Delta table in the schema as Silver--Layer with the table name clean_orders

deduped.write.format('delta').mode('overwrite').saveAsTable('workspace.ecom_data.clean_orders')
deduped.display()

row_id,customer_id,order_date,order_id,sales,_rescued_data
1,CUST272,2024-05-17,ORD1000,296.26,null
2,CUST151,2024-01-02,ORD1001,286.33,null
3,CUST554,2024-09-18,ORD1002,53.17,null
4,CUST061,2024-04-08,ORD1003,198.38,null
5,CUST578,2024-11-06,ORD1004,54.14,null
6,CUST188,2024-06-23,ORD1005,212.11,null
7,CUST670,2024-08-02,ORD1006,23.44,null
8,CUST353,2024-02-04,ORD1007,251.11,null
9,CUST039,2024-06-14,ORD1008,102.7,null
10,CUST253,2024-12-19,ORD1009,143.85,null


In [0]:
%python

from pyspark.sql import functions as F

# Load Silver table

df_silver = spark.table('workspace.ecom_data.clean_orders')

# Check summary statistics
summary = df_silver.agg(
    F.count("*").alias("rows"),
    F.countDistinct("order_id").alias("unique_orders"),
    F.countDistinct("customer_id").alias("customers"),
    F.min("order_date").alias("min_order_date"),
    F.max("order_date").alias("max_order_date")
)
display(summary)

rows,unique_orders,customers,min_order_date,max_order_date
2000,2000,888,2024-01-01,2025-12-31


In [0]:
%python
# Update the Gold-layer cohort_customers in ecom_data Schema with data from 2025

from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_silver = spark.table('workspace.ecom_data.clean_orders')

# assign a unique, sequential number to each order belonging to a specific customer, ordered by the date the purchase was made.

w = Window.partitionBy('customer_id').orderBy('order_date')

# The function F.row_number () assigns the number 1 to the earliest order, 2 to the second, and so on.
orders = df_silver.withColumn('row_num', F.row_number().over(w))

# Since I previously ordered the window by order_date, the row with row_num == 1 is mathematically the earliest purchase for that specific customer and keep only the column which is necessary for this analysis, the customer_id as primary key and renamed these first order dates to first_purchase_date

first_orders = orders.filter(F.col('row_num') == 1).select(
    'customer_id',
    F.col('order_date').alias('first_purchase_date')
)
# It does the same with the second_purchase_date, see above and take the row_num == 2

second_orders = orders.filter(F.col('row_num') == 2).select(
    'customer_id',
    F.col('order_date').alias('second_purchase_date')
)
# Use Left Join for the two tables

cohort = first_orders.join(second_orders, 'customer_id', 'left')

# take the first purchase date as new column cohort_month and it should have only the year and month with the date_trunc function

cohort = cohort.withColumn(
    'cohort_month',
    F.date_trunc('month', F.col('first_purchase_date'))
)

# Calculate the difference between the first and second purchase with the datediff function 

cohort = cohort.withColumn(
    'days_to_second_purchase',
    F.datediff('second_purchase_date', 'first_purchase_date')
)

cohort.write.format('delta').mode('overwrite').saveAsTable('workspace.ecom_data.cohort_customers')
cohort.display()

customer_id,first_purchase_date,second_purchase_date,cohort_month,days_to_second_purchase
CUST001,2024-04-27,null,2024-04-01T00:00:00.000Z,null
CUST002,2024-06-05,2024-10-31,2024-06-01T00:00:00.000Z,148
CUST003,2024-07-25,2024-10-13,2024-07-01T00:00:00.000Z,80
CUST004,2024-09-24,2025-02-22,2024-09-01T00:00:00.000Z,151
CUST007,2024-02-14,2025-04-25,2024-02-01T00:00:00.000Z,436
CUST008,2024-02-17,2024-11-20,2024-02-01T00:00:00.000Z,277
CUST010,2024-11-05,2024-12-10,2024-11-01T00:00:00.000Z,35
CUST011,2024-10-10,null,2024-10-01T00:00:00.000Z,null
CUST012,2024-03-14,2025-05-26,2024-03-01T00:00:00.000Z,438
CUST013,2024-02-23,2024-03-19,2024-02-01T00:00:00.000Z,25


In [0]:
-- Main Validation Query for Gold Layer

WITH validation_stats AS (
  SELECT 
    COUNT(*) AS total_customers,
    -- Count customers where the second purchase date is NULL
    COUNT(CASE WHEN second_purchase_date IS NULL THEN 1 END) AS count_without_second_purchase,
    -- Count customers who HAVE a second purchase date
    COUNT(second_purchase_date) AS count_with_second_purchase,
    -- Check for 2025 data
    MIN(cohort_month) AS earliest_cohort,
    MAX(cohort_month) AS latest_cohort,
    COUNT(CASE WHEN YEAR(cohort_month) = 2025 THEN 1 END) AS rows_in_2025
  FROM workspace.ecom_data.cohort_customers
)
SELECT * FROM validation_stats;


total_customers,count_without_second_purchase,count_with_second_purchase,earliest_cohort,latest_cohort,rows_in_2025
888,308,580,2024-01-01T00:00:00.000Z,2025-12-01T00:00:00.000Z,239


In [0]:
-- Preview a few rows
SELECT * FROM workspace.ecom_data.cohort_customers 
LIMIT 10;

customer_id,first_purchase_date,second_purchase_date,cohort_month,days_to_second_purchase
CUST001,2024-04-27,null,2024-04-01T00:00:00.000Z,null
CUST002,2024-06-05,2024-10-31,2024-06-01T00:00:00.000Z,148
CUST003,2024-07-25,2024-10-13,2024-07-01T00:00:00.000Z,80
CUST004,2024-09-24,2025-02-22,2024-09-01T00:00:00.000Z,151
CUST007,2024-02-14,2025-04-25,2024-02-01T00:00:00.000Z,436
CUST008,2024-02-17,2024-11-20,2024-02-01T00:00:00.000Z,277
CUST010,2024-11-05,2024-12-10,2024-11-01T00:00:00.000Z,35
CUST011,2024-10-10,null,2024-10-01T00:00:00.000Z,null
CUST012,2024-03-14,2025-05-26,2024-03-01T00:00:00.000Z,438
CUST013,2024-02-23,2024-03-19,2024-02-01T00:00:00.000Z,25
